# Практикум 5. Градиентный спуск

Сопровождает интерактивную лабораторию №5 («Градиентный спуск»). Тема курса: оптимизация моделей классификации, функции потерь и сходимость

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.45, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

## Логистическая регрессия «с нуля» на двух признаках
Оставляем два класса и сжимаем текст до 2D, чтобы можно было нарисовать ландшафт потерь по двум весам (w1, w2).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from scipy.special import expit

sub = df[df.label.isin(["спорт", "политика"])]
X = TfidfVectorizer().fit_transform(sub.text)
Z = StandardScaler().fit_transform(TruncatedSVD(2, random_state=0).fit_transform(X))
y = np.where(sub.label == "спорт", 1, -1)
lam = 0.01

def loss(w): return np.mean(np.logaddexp(0, -y * (Z @ w))) + lam / 2 * w @ w
def grad(w, idx=slice(None)):
    s = y[idx] * (Z[idx] @ w)
    return -(Z[idx] * (y[idx] * expit(-s))[:, None]).mean(0) + lam * w

def run(method, lr, steps=60, w0=(-4.0, 4.0), beta=0.9, bs=8, seed=0):
    rng = np.random.default_rng(seed)
    w, v = np.array(w0), np.zeros(2)
    path, losses = [w.copy()], [loss(w)]
    for _ in range(steps):
        if method == "sgd":
            g = grad(w, rng.integers(0, len(y), bs))
        else:
            g = grad(w)
        if method == "momentum":
            v = beta * v + g; w = w - lr * v
        else:
            w = w - lr * g
        path.append(w.copy()); losses.append(loss(w))
    return np.array(path), losses

## 1. Влияние скорости обучения

In [ ]:
for lr in [0.05, 0.5, 2, 8, 40]:
    plt.plot(run("gd", lr)[1], label=f"lr={lr}")
plt.yscale("log"); plt.xlabel("шаг"); plt.ylabel("потери"); plt.legend(); plt.title("Градиентный спуск: сходимость и расхождение"); plt.show()

## 2. Ландшафт потерь и траектории GD, SGD, Momentum

In [ ]:
g = np.linspace(-6, 6, 120)
W1, W2 = np.meshgrid(g, g)
L = np.array([[loss(np.array([a, b])) for a in g] for b in g])
plt.figure(figsize=(6.5, 6))
plt.contourf(W1, W2, L, levels=25, cmap="viridis_r"); plt.colorbar(label="потери")
for m, lr, c in [("gd", 2, "white"), ("sgd", 2, "red"), ("momentum", 0.5, "orange")]:
    p, _ = run(m, lr); plt.plot(p[:, 0], p[:, 1], "-o", ms=3, color=c, label=m)
plt.legend(); plt.xlabel("w1"); plt.ylabel("w2"); plt.title("Траектории оптимизации"); plt.show()

## 3. Сверка с sklearn
Наша цель: среднее значение log-loss плюс `lam/2 * ||w||²`. В sklearn это `C = 1 / (lam * n)`.

In [ ]:
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(fit_intercept=False, C=1 / (lam * len(y)), tol=1e-8).fit(Z, y)
p, _ = run("gd", 2, steps=500)
print("sklearn:", sk.coef_[0].round(3), " наш GD:", p[-1].round(3))

## Задания

1. Подберите `lr`, при котором GD сходится быстрее всего, но не расходится. Как этот `lr` связан с кривизной ландшафта?
2. Измените размер батча `bs` у SGD (1, 8, 64). Как меняются шум траектории и скорость сходимости?
3. Замените функцию потерь на hinge: `np.maximum(0, 1 - y*(Z@w))`. Какой градиент у неё?